# Sentiment Analysis — IMDB Movie Reviews
TF-IDF + Logistic Regression pipeline achieving **89% accuracy** on 22,912 labeled reviews.

In [ ]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:

df = pd.read_csv('IMDB Dataset.csv', engine='python', on_bad_lines='skip')
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

print(f'Dataset size: {len(df)} rows')
print(f'Positive: {df["sentiment"].sum()} | Negative: {len(df) - df["sentiment"].sum()}')

In [ ]:
# ── TEXT CLEANING ─────────────────────────────────────────
def clean_text(text):
    text = re.sub('<.*?>', ' ', text)       # remove HTML tags
    text = re.sub('[^a-zA-Z]', ' ', text)  # keep only letters
    return text.lower().strip()

df['clean_text'] = df['review'].apply(clean_text)
print('Text cleaning done.')

In [ ]:
# ── TF-IDF VECTORIZATION ──────────────────────────────────
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['sentiment']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training on {X_train.shape[0]} samples, testing on {X_test.shape[0]} samples')

In [ ]:
# ── TRAIN & EVALUATE ──────────────────────────────────────
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Model Accuracy: {acc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# ── CONFUSION MATRIX ──────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix — Sentiment Analysis')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print(f'Correctly classified: {cm[0][0] + cm[1][1]} / {len(y_test)}')

In [ ]:
# ── TOP PREDICTIVE WORDS ──────────────────────────────────
feature_names = vectorizer.get_feature_names_out()
coef = model.coef_[0]

top_positive_idx = np.argsort(coef)[-15:]
top_negative_idx = np.argsort(coef)[:15]

print('Top 15 words predicting POSITIVE sentiment:')
for i in top_positive_idx[::-1]:
    print(f'  {feature_names[i]:<20} {coef[i]:.3f}')

print()
print('Top 15 words predicting NEGATIVE sentiment:')
for i in top_negative_idx:
    print(f'  {feature_names[i]:<20} {coef[i]:.3f}')

In [ ]:
# ── PREDICT NEW INPUT ─────────────────────────────────────
def predict_sentiment(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)
    proba = model.predict_proba(vector)[0]
    label = 'Positive' if prediction[0] == 1 else 'Negative'
    confidence = max(proba) * 100
    return f'{label} (confidence: {confidence:.1f}%)'

user_text = input('Enter a sentence: ')
print('Sentiment:', predict_sentiment(user_text))